<a href="https://colab.research.google.com/github/tabassumrafiq/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tabassumrafiq/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
%pip -q install duckdb huggingface_hub

In [12]:
import os
import getpass
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("DuckDB connected successfully.")
print("Tables are ready.")

Paste your Hugging Face READ token (hf_...): ··········
DuckDB connected successfully.
Tables are ready.


In [13]:
for name, src in TABLES.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    count = con.sql(
        f"SELECT COUNT(*) FROM {src}"
    ).fetchone()[0]

    print(f"Rows: {count:,}")


dim_clients
Rows: 104

dim_content
Rows: 519,606

fact_daily


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 78,835,655

fact_daily_sample
Rows: 11,694,072

fact_query_90d
Rows: 2,414,248


In [14]:
for name, src in TABLES.items():
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    schema = con.sql(
        f"DESCRIBE SELECT * FROM {src}"
    ).df()

    print(schema.to_string(index=False))


dim_clients
        column_name column_type null  key default extra
     client_hash_id     VARCHAR  YES None    None  None
          is_active     BOOLEAN  YES None    None  None
     has_gsc_access     BOOLEAN  YES None    None  None
     has_ga4_access     BOOLEAN  YES None    None  None
     access_profile     VARCHAR  YES None    None  None
client_created_date        DATE  YES None    None  None
client_updated_date        DATE  YES None    None  None
     gsc_data_start        DATE  YES None    None  None
     ga4_data_start        DATE  YES None    None  None

dim_content
               column_name column_type null  key default extra
            client_hash_id     VARCHAR  YES None    None  None
           content_hash_id     VARCHAR  YES None    None  None
           keyword_hash_id     VARCHAR  YES None    None  None
               url_hash_id     VARCHAR  YES None    None  None
        keyword_char_count      BIGINT  YES None    None  None
       keyword_token_count      BIGI

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [15]:
# ML-04 — Section 1 verification
# Check the grain and time window for March 2026

grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id) AS unique_content_items,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT CAST(content_hash_id AS VARCHAR) || '|' ||
                      CAST(client_hash_id AS VARCHAR) || '|' ||
                      CAST(report_date AS VARCHAR)) AS unique_content_client_date
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()

print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_content_items  unique_clients  \
0     9841378                331437              55   

   unique_content_client_date  
0                     9841378  


In [16]:
# Verify the March 2026 date span

date_check = con.sql(f"""
    SELECT
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date,
        COUNT(*) AS row_count
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()

print(date_check)

  start_date   end_date  row_count
0 2026-03-01 2026-03-31    9841378


2.** Fields: feature / label / context / excluded**
Sort every field you plan to touch into these four buckets. Excluded needs a why.


### Features

I will use historical performance and query-mix signals that are available before the decision window:

- `gsc_impressions` — previous-period search impressions.
- `gsc_avg_position` — previous-period average search position.
- `content_visible_query_count` — number of visible queries associated with the content.
- `rare_impressions_share` — share of impressions from rare queries.
- `anonymized_impressions_share` — share of impressions from anonymized queries.

### Label

`Needs_Refresh` is the target concept from my ML-03 lane. For this warehouse contract, the label will be defined from a future performance outcome, such as a meaningful decline in search impressions.

### Context

- `client_hash_id` — identifies the client group without exposing the client name.
- `content_hash_id` — identifies the content item without exposing its URL.
- `report_date` — identifies the observation date.
- `month` — identifies the observation month.
- `gsc_data_available` — indicates whether GSC data is available for the observation.

### Excluded

I will exclude future-window performance measures such as `impressions_last30` when they are used to define the label.

**Why:** These values are only known after the prediction period and could directly reveal the outcome. Using them as features would create data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — Verify the grain

This query checks whether the March 2026 slice has one observation for each content item, client, and report date.

In [17]:
# Query 1 — Grain verification

grain_result = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id || '|' ||
                      client_hash_id || '|' ||
                      CAST(report_date AS VARCHAR)) AS unique_content_client_dates
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()

print(grain_result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_content_client_dates
0     9841378                      9841378


### Query 2 — Row count and date span

This query verifies the number of observations and the date range available in the March 2026 slice.

In [18]:
# Query 2 — Count and date-span verification

window_result = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()

print(window_result)

   row_count start_date   end_date
0    9841378 2026-03-01 2026-03-31


### Query 3 — GSC availability

This query checks how many March 2026 observations have GSC data available. The availability condition is explicitly checked with `IS TRUE`.

In [19]:
# Query 3 — Availability verification using IS TRUE

availability_result = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()

print(availability_result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_available_rows
0     9841378             3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

This dataset has several important limitations for the Content Refresh / SEO Performance lane:

1. **Unbalanced client history:** Different clients have different amounts of historical data. This means the same time window may not be available for every client.

2. **GSC availability varies:** Some observations do not have GSC data available. Therefore, search-performance features cannot be treated as complete for every row.

3. **Window overlap:** Historical and future performance windows can overlap across observations. This can make nearby observations less independent.

4. **Observational data:** The warehouse measures webpage performance, but it cannot prove that a content refresh caused an improvement in traffic, clicks, or rankings.

5. **Limited content-quality information:** The data contains performance and content metadata, but it does not fully capture editorial quality, usefulness, or the actual reason a page needs refreshing.

6. **Future information must be excluded:** Outcome-window fields must not be used as input features when defining a future refresh label. Doing so would cause data leakage.

These limitations mean the model should be treated as **decision-support**, not as proof that a page must be refreshed.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Self-check

- [x] Unit of analysis and time window are defined.
- [x] Features, label, context, and excluded fields are identified.
- [x] Three verification queries are included.
- [x] Grain is verified with real warehouse data.
- [x] Row count and date span are verified.
- [x] GSC availability is checked using `IS TRUE`.
- [x] Data limitations are documented.
- [x] Future outcome fields are excluded to avoid leakage.
- [x] No client names, URLs, or private queries are included.
- [x] Claims use careful language such as observed, measured, and decision-support.